In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph


class AgentState(TypedDict):
    query: str
    answer: str

    # 필요한 노드 ? 
    tax_base_equation: str # 과세표준 계산 수식 
    tax_deduction: str  # 공제액 
    market_ratio: str  # 공정시장가액비율
    tax_base: str   # 과세표준 계산
    

graph_builder = StateGraph(AgentState)

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100, separators=["\n\n", "\n"])

In [7]:
from langchain_community.document_loaders import TextLoader

text_path = './docs/real_estate_tax.txt'
loader = TextLoader(text_path,  encoding='utf-8')  # ✅ 인코딩 명시! : 에러 발생함
document_list = loader.load_and_split(text_splitter)

In [ ]:
# 과세표준을 계산하는 방법은 vector 스토어에 있다. retriever 부터 생성하자

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 벡터 스토어 생성 
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


vector_store = Chroma.from_documents(
    documents = document_list,
    embedding=embeddings,
    collection_name='real_estate_tax',
    persist_directory='./real_estate_tax_collection' 
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})  

In [10]:
query = '5억짜리 집 1채, 10억짜리 집 1채, 20억짜리 집 1채를 가지고 있을 때 세금을 얼마나 내나요?'

In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")  # 답변이 잘 안나오는데는 mini 모델을 써서 그럴수도 있다. (원래 gpt-4o 사용)
small_llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
from langchain_classic import hub

rag_prompt = hub.pull('rlm/rag-prompt')



In [19]:
# 수식을 가져오는 노드 생성
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

tax_base_equation_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "사용자의 질문에서 과세표준을 계산하는 방법을 수식으로 나타내주세요. 부연설명 없이 수식만 리턴해주세요."),
        ("human", "{tax_base_equation_information}")
    ]
)

tax_base_retrieval_chain = (
    {'context': retriever, 'question' : RunnablePassthrough()} 
    | rag_prompt 
    | llm 
    | StrOutputParser()
)


tax_base_equation_chain = (
    {'tax_base_equation_information' : RunnablePassthrough()}
    | tax_base_equation_prompt
    | llm
    | StrOutputParser()
)

tax_base_chain = {'tax_base_equation_information' : tax_base_retrieval_chain} | tax_base_equation_chain

def get_tax_base_equation(state: AgentState):
    tax_base_equation_question = '주택에 대한 종합부동산세 계산 시 과세표준을 계산하는 방법을 수식으로 표현해서 알려주세요'
    tax_base_equation = tax_base_chain.invoke(tax_base_equation_question)    # chain 앞에 {'context': retriever} 가 추가되었으므로, context키는 삭제해도됨
    return {'tax_base_equation': tax_base_equation}


In [ ]:
get_tax_base_equation({})

# 생성된 답변은 아래와 같다
# {'tax_base_equation': '주택에 대한 종합부동산세의 과세표준은 납세의무자가 소유한 주택의 공시가격 합산액에서 특정 금액을 공제한 후, 공정시장가액비율을 곱하여 계산합니다. 이 비율은 부동산 시장 동향과 지방 여건을 고려하여 60%에서 100% 사이로 대통령령으로 결정됩니다. 또한, 1세대 1주택자는 12억 원, 법인 또는 법인으로 보는 단체는 6억 원, 일반적으로는 9억 원까지의 합산액이 기준이 됩니다.'}

# 그런데, 줄글로 되어있으면 LLM이 알아듣기 힘들 수 도 있다. 
# 1. 그래서 query 에 "계산하는 방법을 수식으로 표현해서 알려주세요'" 라고 명확히 물어봤지만, 그래도 답변이 수식이 아니라 줄글로 나옴
# 2. 아래 셀을 생각해보자




# {'tax_base_equation': '과세표준 = (주택 공시가격 합산 - 공제금액) × 공정시장가액비율'}

{'tax_base_equation': '과세표준 = (주택 공시가격 합산 - 공제금액) × 공정시장가액비율'}

In [21]:
# 2.
# LLM에 여러 태스크를 한번에 주면, LLM이 헤맨다
# 번역, 요약, 분석에 능하다. 그런데, "분석해서 요약해서 번역해줘 " -> 결과가 안나오는 이유


# chain을 하나 더 만들어서 tax_base_equation 으로 가져온 정보에 기반으로 수식만 추출하게끔 짠다

In [22]:
# 공제액을 찾는 노드


tax_deduction_chain = (
    {'context': retriever, 'question' : RunnablePassthrough()} 
    | rag_prompt 
    | llm 
    | StrOutputParser()
)

def get_tax_deduction(state: AgentState):
    tax_deduction_question = '주택에 대한 종합부동산세 계산 시 공제금액을 알려주세요'
    tax_deduction = tax_deduction_chain.invoke(tax_deduction_question)    # chain 앞에 {'context': retriever} 가 추가되었으므로, context키는 삭제해도됨
    return {'tax_deduction': tax_deduction}


In [ ]:
get_tax_deduction({})


# {'tax_deduction': '주택에 대한 종합부동산세 계산 시 공제금액은 1세대 1주택자의 경우 12억원, 법인 및 법인으로 보는 단체는 6억원, 일반적인 경우 9억원입니다. 또한 만 60세 이상인 1세대 1주택자는 연령에 따라 20%, 30%, 40%의 공제율이 적용됩니다.'}

{'tax_deduction': '주택에 대한 종합부동산세 계산 시 공제금액은 1세대 1주택자의 경우 12억원, 법인 및 법인으로 보는 단체는 6억원, 일반적인 경우 9억원입니다. 또한 만 60세 이상인 1세대 1주택자는 연령에 따라 20%, 30%, 40%의 공제율이 적용됩니다.'}

In [33]:
# 주택에 대한 공정시장가액비율
# 대통령령이라서 웹 서치를 해야함 


# web-search 노드
# from langchain_tavily import TavilySearch
from langchain_community.tools import TavilySearchResults   # cannot import 에러
# from langchain_community.tools import TavilySearch   # cannot import 에러
from datetime import date


tavily_search_tool = TavilySearchResults(
    max_results=5, 
    include_answer=True,
    include_raw_content=True,
    include_images=True,
    search_depth="advanced",
    
)

tax_market_ratio_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", f"아래 정보를 기반으로 공정시장 가액비율을 계산해주세요\n\nContext:\n{{context}}"),
        ("human", "{query}")
    ]
)

def get_market_ratio(state: AgentState):
    query = f'오늘 날짜: ({date.today()})에 해당하는 주택 공시가격 공정시장가액비율은 몇 %인가요'
    context = tavily_search_tool.invoke(query)  
    print(f'context == {context}')
    tax_market_ratio_chain = tax_market_ratio_prompt | llm | StrOutputParser()
    market_ratio = tax_market_ratio_chain.invoke({'context': context, 'query': query})
    return {'market_ratio' : market_ratio}  


In [ ]:
get_market_ratio({})

# {'market_ratio': '2026년 3월 18일 기준 주택의 공정시장가액비율은 60%로 설정되어 있습니다. 이는 법적으로 정해진 범위에 따른 것으로, 특정한 조건에 따라 다른 비율이 적용될 수도 있지만, 기본적인 설정은 60%입니다.'}
# 23년이 gpt 학습날짜 컷오프다. - 지금 24년도니까, 프롬프트에 "날짜정보"를 주도록 하자

context == [{'title': '2026년 상반기 이재명 정부 부동산대책 발표 예정안(+공시가격 ...', 'url': 'https://vejwywc.tistory.com/132', 'content': '체감 보유세가 크게 증가 할 수 밖에 없습니다.\n\n실제 세금 변화는 어느 정도일까요?\n\n신한은행 모의 계산에 따르면\n\n공시가격과 공정시장가액비율이\n\n함께 오를 경우, 보유세가 최대 50%\n\n가까이 급증할 것으로 나타났습니다.\n\n대표적인 사례로\n\n서초구 반포자이 전용 84의\n\n25년 공시가격은\n\n약 27억7400만원입니다.\n\n이 아파트를 1주택자가 보유할 경우\n\n2025년 보유세\n\n(재산세+종부세): 약 1274만원\n\n2026년 예상 보유세\n\n(공정시장가액비율 동결 시)\n\n약 1572만원 (+23.4%)\n\n2026년 예상 보유세\n\n(공정시장가액비율 80% 상향 시)\n\n약 1842만원 (+44.6%)\n\n(568만원 증가하게 되는 셈)\n\n공정시장가액비율은 법적으로\n\n60~100% 범위에서 시행령으로\n\n조정할 수 있습니다.\n\n문재인 정부, 80~95%까지 인상\n\n윤석열 정부 초기\n\n2022년부터 종부세 60%\n\n재산세 45%로 하향 조정\n\n하지만 최근 집값이\n\n강남을 중심으로 다시 급등하고,\n\n2025년 종부세 수입이\n\n1조6000억 원으로 급감하면서\n\n정부는 세수 확보를 위한 수단으로\n\n공정시장가액비율 인상 카드를\n\n꺼내든 것으로 보입니다.\n\n### 공정시장가액 적용 범위와 현행\n\n재산세\n\n주택은 40~80% 범위에서\n\n대통령령으로 정함.\n\n(1세대1주택은 30~70% 특례 적용)\n\n2023년 1세대1주택 재산세\n\n공정시장가액비율\n\n3억 이하 43%\n\n3억 초과~6억 이하 44%\n\n6억 초과 45%\n\n(다주택자·법인은 60%가 적용)\n\n주택은 60~100% 범위에서 시행령으로 정하며 [...] 부동

{'market_ratio': '2026년 3월 18일 기준 주택의 공정시장가액비율은 60%로 설정되어 있습니다. 이는 법적으로 정해진 범위에 따른 것으로, 특정한 조건에 따라 다른 비율이 적용될 수도 있지만, 기본적인 설정은 60%입니다.'}

In [38]:
from langchain_core.prompts import PromptTemplate

tax_base_caculation_prompt = PromptTemplate.from_template("""
주어진 내용을 기반으로 과세표준을 계산해주세요
                                                          
과세표준 계산 공식: {tax_base_equation}
공제금액: {tax_deduction}
공정시장가액비율: {market_ratio}
사용자 주택 공시가격 정보 : {query}""")

def calculate_tax_base(state: AgentState):
    tax_base_equation = state['tax_base_equation']
    tax_deduction = state['tax_deduction']
    market_ratio = state['market_ratio']

    query = state['query']

    # 체인 생성
    tax_base_caculation_chain =    tax_base_caculation_prompt | llm | StrOutputParser()
    tax_base = tax_base_caculation_chain.invoke({
        'tax_base_equation': tax_base_equation,
        'tax_deduction': tax_deduction,
        'market_ratio': market_ratio,
        'query': query
    })
    print(f'tax_base == {tax_base}')
    return {'tax_base': tax_base}


In [39]:
initial_state = {
    'query': query,
    'tax_base_equation': '과세표준 = (주택 공시가격 합산 - 공제금액) × 공정시장가액비율',
    'tax_deduction': '주택에 대한 종합부동산세 계산 시 공제금액은 1세대 1주택자의 경우 12억원, 법인 및 법인으로 보는 단체는 6억원, 일반적인 경우 9억원입니다. 또한 만 60세 이상인 1세대 1주택자는 연령에 따라 20%, 30%, 40%의 공제율이 적용됩니다.',
    'market_ratio': '2026년 3월 18일 기준 주택의 공정시장가액비율은 60%로 설정되어 있습니다. 이는 법적으로 정해진 범위에 따른 것으로, 특정한 조건에 따라 다른 비율이 적용될 수도 있지만, 기본적인 설정은 60%입니다.'
    
    }

In [40]:
calculate_tax_base(initial_state)

tax_base == 주어진 정보를 기반으로 과세표준을 계산해보겠습니다.

1. **주택 공시가격 합산**:
   - 5억 + 10억 + 20억 = 35억

2. **공제금액**:
   - 1세대 1주택자가 아닌 일반적인 경우(여기서는 3채의 집이 있으므로 일반으로 계산) 따라서 공제금액은 9억원입니다.

3. **과세표준 계산**:
   - 과세표준 = (주택 공시가격 합산 - 공제금액) × 공정시장가액비율
   - 과세표준 = (35억 - 9억) × 0.6
   - 과세표준 = 26억 × 0.6
   - 과세표준 = 15.6억

따라서, 최종적으로 계산된 과세표준은 **15.6억 원**입니다.


{'tax_base': '주어진 정보를 기반으로 과세표준을 계산해보겠습니다.\n\n1. **주택 공시가격 합산**:\n   - 5억 + 10억 + 20억 = 35억\n\n2. **공제금액**:\n   - 1세대 1주택자가 아닌 일반적인 경우(여기서는 3채의 집이 있으므로 일반으로 계산) 따라서 공제금액은 9억원입니다.\n\n3. **과세표준 계산**:\n   - 과세표준 = (주택 공시가격 합산 - 공제금액) × 공정시장가액비율\n   - 과세표준 = (35억 - 9억) × 0.6\n   - 과세표준 = 26억 × 0.6\n   - 과세표준 = 15.6억\n\n따라서, 최종적으로 계산된 과세표준은 **15.6억 원**입니다.'}

In [ ]:
def calculate_tax_rate(state: AgentState):
    